In [23]:
import pandas as pd
from pathlib import Path

DF_PATH = Path("../data/raw/paysim dataset.csv")
df = pd.read_csv(DF_PATH, dtype={
    "amount": "float32",
    "oldbalanceOrg": "float32",
    "newbalanceOrig": "float32",
    "oldbalanceDest": "float32",
    "newbalanceDest": "float32",
    "isFraud": "int8",
    "isFlaggedFraud": "int8",
})

In [24]:
# look at the dataframe
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.639648,C1231006815,170136.0,160296.359375,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.280029,C1666544295,21249.0,19384.720703,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.000000,C1305486145,181.0,0.000000,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.000000,C840083671,181.0,0.000000,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.139648,C2048537720,41554.0,29885.859375,M1230701703,0.0,0.0,0,0


In [25]:
# shape
df.shape

(6362620, 11)

In [26]:
# info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float32
 3   nameOrig        str    
 4   oldbalanceOrg   float32
 5   newbalanceOrig  float32
 6   nameDest        str    
 7   oldbalanceDest  float32
 8   newbalanceDest  float32
 9   isFraud         int8   
 10  isFlaggedFraud  int8   
dtypes: float32(5), int64(1), int8(2), str(3)
memory usage: 499.9 MB


In [27]:
# drop account identifiers
df.drop(columns=["nameOrig", "nameDest"], inplace=True)

In [28]:
# rename some columns
df = df.rename(columns={"oldbalanceOrg": "oldBalanceOrig", "newbalanceOrig": "newBalanceOrig", 
                        "oldbalanceDest": "oldBalanceDest", "newbalanceDest": "newBalanceDest"})

In [29]:
# check duplicate values
df.duplicated().sum()

np.int64(543)

In [30]:
# check NaN values
df.isna().sum()

step              0
type              0
amount            0
oldBalanceOrig    0
newBalanceOrig    0
oldBalanceDest    0
newBalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [31]:
# fix type values
df.type = df.type.str.replace("_", " ")

# change "type" data type for better performance
df.type = df.type.astype("category")

In [32]:
# check that there are no negative amounts
df.amount.min()

np.float32(0.0)

In [33]:
# check that balances are non-negative
balance_cols = ["oldBalanceOrig", "newBalanceOrig", "oldBalanceDest", "newBalanceDest"]
(df[balance_cols] < 0).sum()

oldBalanceOrig    0
newBalanceOrig    0
oldBalanceDest    0
newBalanceDest    0
dtype: int64

In [34]:
# move target to end
col_to_move = df.pop("isFraud")
df.insert(len(df.columns), "isFraud", col_to_move)

In [35]:
# see final df
df.head()

,step,type,amount,oldBalanceOrig,newBalanceOrig,oldBalanceDest,newBalanceDest,isFlaggedFraud,isFraud
0,1,PAYMENT,9839.639648,170136.0,160296.359375,0.0,0.0,0,0
1,1,PAYMENT,1864.280029,21249.0,19384.720703,0.0,0.0,0,0
2,1,TRANSFER,181.000000,181.0,0.000000,0.0,0.0,0,1
3,1,CASH OUT,181.000000,181.0,0.000000,21182.0,0.0,0,1
4,1,PAYMENT,11668.139648,41554.0,29885.859375,0.0,0.0,0,0


In [36]:
# save the cleaned dataframe
df.to_parquet("../data/processed/paysim_cleaned.parquet", index=False)